# AI Spot Trading — Dataset Builder

Builds the training/validation/agent arrays once and publishes them as a Kaggle Dataset.
Both the **Colab** and **Kaggle** training notebooks then import this dataset instead of
downloading + rebuilding features every run.

## How to publish
1. Run this notebook end-to-end on **Kaggle** (no GPU required).
2. After it finishes, `dataset.npz` lands in `/kaggle/working/`.
3. From the notebook **Output**, "Create new dataset" with slug `acaurangel/ai-spot-trading-dataset`.
4. New versions: run again, then publish a **new version** of the same dataset.

In [1]:
!pip install -q ccxt PyWavelets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.4/145.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 11.7 MB/s eta 0:00:00


In [ ]:
import time, os
import numpy as np
import ccxt

# Dataset-relevant hyperparameters (MUST match the training notebooks)
WINDOW_SIZE          = 128            # 128 x 15min ~ 32h of context
HORIZON              = 4              # predict direction 4 x 15min = 60 min ahead
NUM_FEATURES         = 20             # 10 original + 6 wavelet + 3 liquidity + 1 ADX

# Constants
INDICATOR_WARMUP      = 200
RETURN_CLIP           = 0.1
VOLATILITY_WINDOW     = 20
BACKTEST_HOLDOUT      = 1000
VALIDATION_FRACTION   = 0.1
EMBARGO_FRACTION      = 0.01
HISTORICAL_CANDLES    = 35_000        # 35k x 15min ~ 365 days

STOP_LOSS_PCT    = 0.0030
TAKE_PROFIT_PCT  = 0.0050
SLIPPAGE_PCT     = 0.0005

SOFT_LABEL_SCALE     = 400            # MUST match the training notebooks
SOFT_LABEL_CLIP      = (0.02, 0.98)

FORECASTER_SYMBOLS = [
    'BTC/USDT', 'ETH/USDT',
    'SOL/USDT', 'LINK/USDT', 'BNB/USDT',
    'XRP/USDT', 'DOGE/USDT'
]
AGENT_SYMBOL = 'BTC/USDT'
TIMEFRAME    = '15m'

if os.path.isdir('/kaggle/working'):
    WORK_DIR = '/kaggle/working'
elif os.path.isdir('/content'):
    WORK_DIR = '/content'
else:
    WORK_DIR = '.'
print(f'WORK_DIR = {WORK_DIR}')

WORK_DIR = /kaggle/working


In [3]:
# ── Technical indicators (pure NumPy) ────────────────────────────────────────

def sma(values: np.ndarray, period: int) -> np.ndarray:
    res = np.full(len(values), np.nan)
    if len(values) < period: return res
    window_sum = np.sum(values[:period])
    res[period - 1] = window_sum / period
    for i in range(period, len(values)):
        window_sum += values[i] - values[i - period]
        res[i] = window_sum / period
    return res

def ema(values: np.ndarray, period: int) -> np.ndarray:
    alpha = 2.0 / (period + 1)
    result = np.full(len(values), np.nan)
    if len(values) < period: return result
    result[period - 1] = np.mean(values[:period])
    for i in range(period, len(values)):
        result[i] = values[i] * alpha + result[i - 1] * (1 - alpha)
    return result

def rsi(values: np.ndarray, period: int = 14) -> np.ndarray:
    deltas = np.diff(values)
    result = np.full(len(values), np.nan)
    if len(deltas) < period: return result
    gains = np.where(deltas > 0, deltas, 0.0)
    losses = np.where(deltas < 0, -deltas, 0.0)
    avg_g, avg_l = np.mean(gains[:period]), np.mean(losses[:period])
    rs = avg_g / (avg_l + 1e-10)
    result[period] = 100 - 100 / (1 + rs)
    for i in range(period, len(deltas)):
        avg_g = (avg_g * (period - 1) + gains[i]) / period
        avg_l = (avg_l * (period - 1) + losses[i]) / period
        rs = avg_g / (avg_l + 1e-10)
        result[i + 1] = 100 - 100 / (1 + rs)
    return result

def macd(values: np.ndarray, fast=12, slow=26, signal=9):
    fast_ema = ema(values, fast)
    slow_ema = ema(values, slow)
    macd_line = fast_ema - slow_ema
    signal_line = ema(np.where(np.isnan(macd_line), 0, macd_line), signal)
    return macd_line, signal_line

def bollinger(values: np.ndarray, period: int = 20, std_dev: float = 2.0):
    upper = np.full(len(values), np.nan)
    lower = np.full(len(values), np.nan)
    for i in range(period - 1, len(values)):
        window = values[i - period + 1 : i + 1]
        m = np.mean(window)
        s = np.std(window, ddof=0)
        upper[i] = m + std_dev * s
        lower[i] = m - std_dev * s
    return upper, lower

def rolling_vwap(highs: np.ndarray, lows: np.ndarray, closes: np.ndarray, volumes: np.ndarray, period: int = 96) -> np.ndarray:
    """Rolling VWAP over 'period' candles. 96 * 15m = 24 hours."""
    typical_price = (highs + lows + closes) / 3.0
    pv = typical_price * volumes
    vwap = np.full(len(closes), np.nan)
    if len(closes) < period: return vwap
    sum_pv = np.sum(pv[:period])
    sum_v = np.sum(volumes[:period])
    vwap[period - 1] = sum_pv / (sum_v + 1e-10)
    for i in range(period, len(closes)):
        sum_pv += pv[i] - pv[i - period]
        sum_v += volumes[i] - volumes[i - period]
        vwap[i] = sum_pv / (sum_v + 1e-10)
    return vwap

def obv(closes: np.ndarray, volumes: np.ndarray) -> np.ndarray:
    """On-Balance Volume."""
    res = np.zeros(len(closes))
    res[0] = volumes[0]
    for i in range(1, len(closes)):
        if closes[i] > closes[i-1]:
            res[i] = res[i-1] + volumes[i]
        elif closes[i] < closes[i-1]:
            res[i] = res[i-1] - volumes[i]
        else:
            res[i] = res[i-1]
    return res

def adx_approximation(highs: np.ndarray, lows: np.ndarray, closes: np.ndarray, period: int = 14) -> np.ndarray:
    up_move = np.diff(highs)
    down_move = np.diff(lows) * -1
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    tr1 = highs[1:] - lows[1:]
    tr2 = np.abs(highs[1:] - closes[:-1])
    tr3 = np.abs(lows[1:] - closes[:-1])
    tr = np.maximum(tr1, np.maximum(tr2, tr3))
    atr = ema(tr, period)
    plus_di = 100 * (ema(plus_dm, period) / (atr + 1e-10))
    minus_di = 100 * (ema(minus_dm, period) / (atr + 1e-10))
    dx = 100 * np.abs(plus_di - minus_di) / (plus_di + minus_di + 1e-10)
    adx = ema(dx, period)
    return np.concatenate(([np.nan], adx))

print("Indicators OK.")

# ── Wavelet decomposition (dual-path frequency features) ─────────────────────
import pywt

WAVELET_NAME  = 'db4'
WAVELET_LEVEL = 2

def wavelet_decompose_series(closes: np.ndarray) -> dict:
    n = len(closes)
    pad_len = int(2 ** np.ceil(np.log2(n)))
    padded = np.pad(closes, (0, pad_len - n), mode='edge')
    coeffs = pywt.wavedec(padded, WAVELET_NAME, level=WAVELET_LEVEL)
    trend_coeffs = [coeffs[0]] + [np.zeros_like(c) for c in coeffs[1:]]
    trend = pywt.waverec(trend_coeffs, WAVELET_NAME)[:n]
    med_coeffs = [np.zeros_like(coeffs[0]), coeffs[1]] + [np.zeros_like(c) for c in coeffs[2:]]
    medium = pywt.waverec(med_coeffs, WAVELET_NAME)[:n]
    high_coeffs = [np.zeros_like(coeffs[0])] + [np.zeros_like(c) for c in coeffs[1:-1]] + [coeffs[-1]]
    high = pywt.waverec(high_coeffs, WAVELET_NAME)[:n]
    return {'trend': trend, 'medium': medium, 'high': high}

def wavelet_energy_ratio(detail: np.ndarray, window: int = 20) -> np.ndarray:
    energy = np.zeros(len(detail))
    for i in range(window, len(detail)):
        segment = detail[i - window : i]
        energy[i] = np.mean(segment ** 2)
    return energy

def wavelet_trend_slope(trend: np.ndarray, window: int = 10) -> np.ndarray:
    slope = np.zeros(len(trend))
    for i in range(window, len(trend)):
        segment = trend[i - window : i]
        x = np.arange(window)
        slope[i] = np.polyfit(x, segment, 1)[0]
    return slope

def spectral_entropy_rolling(closes: np.ndarray, window: int = 32) -> np.ndarray:
    entropy = np.zeros(len(closes))
    for i in range(window, len(closes)):
        segment = closes[i - window : i]
        fft_amp = np.abs(np.fft.rfft(segment - np.mean(segment)))
        psd = fft_amp ** 2
        psd_norm = psd / (np.sum(psd) + 1e-12)
        entropy[i] = -np.sum(psd_norm * np.log2(psd_norm + 1e-12))
    max_ent = np.log2(window // 2 + 1)
    entropy = entropy / max_ent
    return entropy

print("Wavelet decomposition + spectral entropy OK.")

Indicators OK.
Wavelet decomposition + spectral entropy OK.


In [4]:
# ── Candle fetch — multi-exchange with fallback ───────────────────────────────

TIMEFRAME_MS = {
    "1m": 60_000, "3m": 180_000, "5m": 300_000, "15m": 900_000,
    "30m": 1_800_000, "1h": 3_600_000, "2h": 7_200_000, "4h": 14_400_000,
    "6h": 21_600_000, "8h": 28_800_000, "12h": 43_200_000, "1d": 86_400_000,
}
MAX_PER_REQUEST = 1000

EXCHANGE_CONFIGS = [
    (lambda: ccxt.bybit({"enableRateLimit": True}),  {"category": "spot"}),
    (lambda: ccxt.okx({"enableRateLimit": True}),    {}),
    (lambda: ccxt.kucoin({"enableRateLimit": True}), {}),
    (lambda: ccxt.gate({"enableRateLimit": True}),   {}),
    (lambda: ccxt.mexc({"enableRateLimit": True}),   {}),
]

_active_exchange = None
_active_params   = {}

def _get_exchange():
    global _active_exchange, _active_params
    for factory, extra_params in EXCHANGE_CONFIGS:
        ex = factory()
        try:
            ex.load_markets()
            if "BTC/USDT" not in ex.markets:
                print(f"  x {ex.id} -- BTC/USDT not found")
                continue
            test = ex.fetch_ohlcv("BTC/USDT", "1h", limit=3, params=extra_params)
            if not test or len(test) == 0:
                print(f"  x {ex.id} -- fetch_ohlcv returned empty")
                continue
            print(f"  v Active exchange: {ex.id} (smoke-test OK: {len(test)} candles)")
            _active_exchange = ex
            _active_params   = extra_params
            return ex, extra_params
        except Exception as e:
            print(f"  x {ex.id} unavailable: {type(e).__name__}: {str(e)[:80]}")
    raise RuntimeError("No exchange accessible. Check Kaggle network connectivity.")

print("Searching for available exchange...")
_active_exchange, _active_params = _get_exchange()

# Diagnose available pairs
for sym in FORECASTER_SYMBOLS:
    available = sym in _active_exchange.markets
    print(f"  {sym}: {'OK' if available else 'NOT FOUND'}")

def fetch_candles(symbol: str, limit: int, timeframe: str = "1h") -> np.ndarray:
    """Returns array of shape (N, 6): [ts, open, high, low, close, volume]."""
    global _active_exchange, _active_params
    ex     = _active_exchange
    params = _active_params

    if symbol not in ex.markets:
        print(f"  ! {symbol} not found in {ex.id}")
        return np.empty((0, 6), dtype=np.float64)

    tf_ms      = TIMEFRAME_MS[timeframe]
    end_time   = int(time.time() * 1000)
    start_time = end_time - limit * tf_ms
    all_rows   = []
    since      = start_time

    while since < end_time:
        try:
            raw = ex.fetch_ohlcv(symbol, timeframe, since=since,
                                  limit=MAX_PER_REQUEST, params=params)
        except Exception as e:
            print(f"  ! Error on {ex.id} ({type(e).__name__}), reconnecting...")
            ex, params = _get_exchange()
            continue

        if not raw:
            break

        filtered = [row for row in raw if row[0] <= end_time]
        all_rows.extend(filtered)

        # Advance cursor by timestamp, NOT by batch size
        next_since = raw[-1][0] + tf_ms
        if next_since <= since:
            break
        since = next_since

        if len(all_rows) >= limit:
            break

        time.sleep(ex.rateLimit / 1000)

    if len(all_rows) == 0:
        print(f"  ! WARNING: 0 candles returned for {symbol}!")
        return np.empty((0, 6), dtype=np.float64)

    arr = np.array(all_rows, dtype=np.float64)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)

    _, idx = np.unique(arr[:, 0], return_index=True)
    arr = arr[idx]

    print(f"  -> {symbol}: {len(arr)} candles downloaded (requested: {limit})")
    return arr[-limit:]

print("Fetch function ready.")


Searching for available exchange...
  x bybit unavailable: RateLimitExceeded: bybit GET https://api.bybit.com/v5/market/instruments-info?category=spot 403 For
  v Active exchange: okx (smoke-test OK: 3 candles)
  BTC/USDT: OK
  ETH/USDT: OK
  SOL/USDT: OK
  LINK/USDT: OK
  BNB/USDT: OK
  XRP/USDT: OK
  DOGE/USDT: OK
Fetch function ready.


In [5]:
# Feature builder (19 features per timestep) + SOFT direction targets

def soft_label(future_return: float, scale: float = SOFT_LABEL_SCALE,
               clip_lo: float = SOFT_LABEL_CLIP[0],
               clip_hi: float = SOFT_LABEL_CLIP[1]) -> float:
    sig = 1.0 / (1.0 + np.exp(-future_return * scale))
    return float(np.clip(sig, clip_lo, clip_hi))

def build_features_for_series(candles: np.ndarray, window_size: int = 128) -> tuple:
    closes  = candles[:, 4]
    opens   = candles[:, 1]
    highs   = candles[:, 2]
    lows    = candles[:, 3]
    volumes = candles[:, 5]

    # ── Original indicators (computed globally, fully causal) ──────────
    ema9_all   = ema(closes, 9)
    ema21_all  = ema(closes, 21)
    rsi14_all  = rsi(closes, 14)
    macd_line, _ = macd(closes, 12, 26, 9)
    bb_upper, bb_lower = bollinger(closes, 20, 2.0)
    sp_entropy = spectral_entropy_rolling(closes, window=32)
    adx_all = adx_approximation(highs, lows, closes, 14)

    # ── Liquidity & Volume indicators (NEW) ──────────
    vwap_all = rolling_vwap(highs, lows, closes, volumes, 96) # 24h rolling VWAP
    obv_all  = obv(closes, volumes)
    obv_ema  = ema(obv_all, 21)
    vol_sma  = sma(volumes, 20)

    X_list, y_list = [], []
    n = len(candles)
    mean_bar_return = float(np.mean(np.diff(closes) / closes[:-1]))

    for i in range(window_size, n - HORIZON):
        w_start = i - window_size
        w_end   = i

        ref_close  = closes[w_start]
        max_volume = np.max(volumes[w_start:w_end]) if np.max(volumes[w_start:w_end]) > 0 else 1.0
        if max_volume == 0: max_volume = 1.0

        window_closes = closes[w_start:w_end]
        wv = wavelet_decompose_series(window_closes)
        wv_trend   = wv['trend']
        wv_medium  = wv['medium']
        wv_high    = wv['high']
        wv_slope   = wavelet_trend_slope(wv_trend, window=10)
        wv_med_nrg = wavelet_energy_ratio(wv_medium, window=20)
        wv_hi_nrg  = wavelet_energy_ratio(wv_high, window=20)

        slope_std = np.std(wv_slope) + 1e-10

        rows = []
        for j in range(w_start, w_end):
            local_j = j - w_start
            c = closes[j]
            prev_ret = 0.0 if j == w_start else (c - closes[j-1]) / closes[j-1]
            
            r9   = ema9_all[j]   if not np.isnan(ema9_all[j])   else c
            r21  = ema21_all[j]  if not np.isnan(ema21_all[j])  else c
            rsi_v = rsi14_all[j] if not np.isnan(rsi14_all[j])  else 50.0
            macd_v = macd_line[j] if not np.isnan(macd_line[j]) else 0.0
            bbu  = bb_upper[j]   if not np.isnan(bb_upper[j])   else c
            bbl  = bb_lower[j]   if not np.isnan(bb_lower[j])   else c
            adx_val = adx_all[j] if not np.isnan(adx_all[j])    else 20.0
            
            # Safe fallbacks for new features
            vwp = vwap_all[j] if not np.isnan(vwap_all[j]) else c
            vsma = vol_sma[j] if not np.isnan(vol_sma[j]) else (volumes[j] + 1e-8)
            oe21 = obv_ema[j] if not np.isnan(obv_ema[j]) else obv_all[j]

            med_e = wv_med_nrg[local_j] if wv_med_nrg[local_j] > 0 else 1e-6
            hi_e  = wv_hi_nrg[local_j]  if wv_hi_nrg[local_j] > 0  else 0.0
            noise_to_signal = float(np.clip(hi_e / med_e, 0.0, 10.0))

            # Liquidity features calcs
            vwap_dist = (c - vwp) / vwp  # Positivo = Bullish (acima da VWAP)
            rel_vol = volumes[j] / (vsma + 1e-8)
            # Divergência do OBV normalizada em "unidades de barras médias de volume"
            obv_oscillator = (obv_all[j] - oe21) / (vsma + 1e-8)

            rows.append([
                c / ref_close - 1,               # 0
                volumes[j] / max_volume,         # 1
                rsi_v / 100.0,                   # 2
                macd_v / c,                      # 3
                prev_ret,                        # 4
                (highs[j] - lows[j]) / c,        # 5
                (r9 - c) / c,                    # 6
                (r21 - c) / c,                   # 7
                (bbu - c) / c,                   # 8
                (bbl - c) / c,                   # 9
                (wv_trend[local_j] - c) / c,     # 10
                wv_medium[local_j] / c,          # 11
                wv_high[local_j] / c,            # 12
                wv_slope[local_j] / slope_std,   # 13
                noise_to_signal,                 # 14
                sp_entropy[j],                   # 15
                vwap_dist,                       # 16: VWAP distance (Liquidez)
                rel_vol,                         # 17: Relative volume (Liquidez)
                obv_oscillator,                  # 18: OBV oscillator (Liquidez)
                adx_val / 100.0,                 # 19: ADX normalizado (Novo)
            ])

        X_list.append(rows)

        anchor_close = closes[i]
        future = []
        for h in range(1, HORIZON + 1):
            future_return = (closes[i + h] - anchor_close) / anchor_close
            excess_return = future_return - mean_bar_return * h
            future.append(soft_label(excess_return))
        y_list.append(future)

    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

def compute_volatility(candles: np.ndarray) -> np.ndarray:
    closes = candles[:, 4]
    n = len(closes)
    vol = np.zeros(n)
    for i in range(1, n):
        start = max(1, i - VOLATILITY_WINDOW)
        rets = (closes[start:i+1] - closes[start-1:i]) / closes[start-1:i]
        vol[i] = np.std(rets)
    return vol

print(f"Feature builder OK: {NUM_FEATURES} features, soft labels (scale={SOFT_LABEL_SCALE}).")

Feature builder OK: 20 features, soft labels (scale=400).


In [6]:
# ── Purged Embargoed Split ───────────────────────────────────────────────────

def purged_embargoed_split(X, y, horizon, embargo_frac, val_frac=0.1):
    total = len(X)
    val_size   = int(total * val_frac)
    val_start  = total - val_size
    embargo_sz = int(total * embargo_frac)
    train_end  = max(0, val_start - horizon - embargo_sz)
    purged = val_start - train_end
    return (
        X[:train_end], y[:train_end],
        X[val_start:],  y[val_start:],
        purged
    )

print("Split OK.")


Split OK.


In [7]:
# ── Download and preprocessing ───────────────────────────────────────────────

total_needed = HISTORICAL_CANDLES + BACKTEST_HOLDOUT

all_X_train, all_y_train = [], []
all_X_val,   all_y_val   = [], []
agent_candles = None

for sym in FORECASTER_SYMBOLS:
    print(f"\nDownloading {total_needed} candles for {sym}...")
    raw = fetch_candles(sym, total_needed, TIMEFRAME)
    train_raw = raw[:-BACKTEST_HOLDOUT] if len(raw) > BACKTEST_HOLDOUT else raw

    print(f"  {len(train_raw)} candles for training")

    min_needed = WINDOW_SIZE + HORIZON + 1
    if len(train_raw) < min_needed:
        print(f"  ! Insufficient data (minimum: {min_needed}) -- skipping.")
        continue

    if sym == AGENT_SYMBOL:
        agent_candles = train_raw.copy()

    X, y = build_features_for_series(train_raw, WINDOW_SIZE)
    Xtr, ytr, Xv, yv, purged = purged_embargoed_split(
        X, y, HORIZON, EMBARGO_FRACTION, VALIDATION_FRACTION
    )
    all_X_train.append(Xtr);  all_y_train.append(ytr)
    all_X_val.append(Xv);     all_y_val.append(yv)
    print(f"  train={len(Xtr)}, val={len(Xv)}, purged={purged}")

if not all_X_train:
    raise RuntimeError("No symbol returned data.")

if agent_candles is None:
    raise RuntimeError(f"No data for agent ({AGENT_SYMBOL}).")

X_train = np.concatenate(all_X_train, axis=0)
y_train = np.concatenate(all_y_train, axis=0)
X_val   = np.concatenate(all_X_val,   axis=0)
y_val   = np.concatenate(all_y_val,   axis=0)

print(f"\nTotal dataset -- train: {X_train.shape}, val: {X_val.shape}")



  -> BTC/USDT: 36000 candles downloaded (requested: 36000)
  35000 candles for training
  train=31030, val=3486, purged=352

  -> ETH/USDT: 36000 candles downloaded (requested: 36000)
  35000 candles for training
  train=31030, val=3486, purged=352

  -> SOL/USDT: 36000 candles downloaded (requested: 36000)
  35000 candles for training
  train=31030, val=3486, purged=352

  -> LINK/USDT: 36000 candles downloaded (requested: 36000)
  35000 candles for training
  train=31030, val=3486, purged=352

  -> BNB/USDT: 36000 candles downloaded (requested: 36000)
  35000 candles for training
  train=31030, val=3486, purged=352

  -> XRP/USDT: 36000 candles downloaded (requested: 36000)
  35000 candles for training
  train=31030, val=3486, purged=352

  -> DOGE/USDT: 36000 candles downloaded (requested: 36000)
  35000 candles for training
  train=31030, val=3486, purged=352

Total dataset -- train: (217210, 128, 20), val: (24402, 128, 20)


In [8]:
# ── Pre-compute agent features and volatility ────────────────────────────────

print("Pre-computing agent features...")
agent_X, _ = build_features_for_series(agent_candles, WINDOW_SIZE)
agent_vol  = compute_volatility(agent_candles)
agent_vol_samples = agent_vol[WINDOW_SIZE : len(agent_candles) - HORIZON]
agent_adx_full = adx_approximation(agent_candles[:, 2], agent_candles[:, 3], agent_candles[:, 4], 14)
agent_adx_samples = agent_adx_full[WINDOW_SIZE : len(agent_candles) - HORIZON]

print(f"Agent features: {agent_X.shape}, volatility: {agent_vol_samples.shape}")


Pre-computing agent features...
Agent features: (34868, 128, 20), volatility: (34868,)


In [9]:
# Save the dataset bundle. Single .npz with everything the training notebooks need.
# Publish the resulting file as the Kaggle Dataset acaurangel/ai-spot-trading-dataset.

OUT = f'{WORK_DIR}/dataset.npz'
np.savez_compressed(
    OUT,
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    agent_X=agent_X, agent_candles=agent_candles, 
    agent_vol_samples=agent_vol_samples,
    agent_adx_samples=agent_adx_samples,
)
print(f'Saved {OUT} | size: {os.path.getsize(OUT)/1e6:.1f} MB')
print()
print('Shapes:')
print(f'  X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'  X_val:   {X_val.shape},  y_val:   {y_val.shape}')
print(f'  agent_X: {agent_X.shape}, agent_candles: {agent_candles.shape}, agent_vol: {agent_vol_samples.shape}')
print()
print('Next: Notebook output panel -> Create new dataset -> slug acaurangel/ai-spot-trading-dataset')

Saved /kaggle/working/dataset.npz | size: 905.5 MB

Shapes:
  X_train: (217210, 128, 20), y_train: (217210, 4)
  X_val:   (24402, 128, 20),  y_val:   (24402, 4)
  agent_X: (34868, 128, 20), agent_candles: (35000, 6), agent_vol: (34868,)

Next: Notebook output panel -> Create new dataset -> slug acaurangel/ai-spot-trading-dataset
